In [0]:
%run ./utils

In [0]:
from datetime import datetime, timedelta
import pyspark.sql.functions as F

# 默认监控的 project 名称。
# 当前监控目标为 consumerlist 这个 Workflow 的日志。
DEFAULT_PROJECT = "consumerlist"

# 默认监控的 step_num 列表，对应 Consumer Pipeline 的主流程步骤。
# 只有这些步骤的日志会被纳入无日志判断。
DEFAULT_STEP_NUM_LIST = [
    "01-2", "01",
    "02",
    "03-1", "03-2", "03-3", "03-4",
    "04-1", "04-2", "04-3", "04-4",
    "05-1", "05-2", "05-3", "05-4",
    "06-1", "06-2", "06-3",
    "07",
]

# 默认无日志告警阈值（小时）。
# 当前脚本在 Jobs 中的触发频率为 hourly。
DEFAULT_NO_LOG_HOURS = 2

In [0]:
def parse_step_num_list(step_num_list_str: str | None) -> list[str]:
    """
    解析逗号分隔的 step_num 字符串。

    参数:
        step_num_list_str: 例如 "01,02,03-1"。为空或 None 时使用 DEFAULT_STEP_NUM_LIST。
    返回:
        去空白后的 step_num 列表。
    """
    if not step_num_list_str or not step_num_list_str.strip():
        return DEFAULT_STEP_NUM_LIST
    return [x.strip() for x in step_num_list_str.split(",") if x.strip()]


def build_email_body(
    monitor_id: str,
    end_time: datetime,
    no_log_hours: int,
    step_num_list: list[str],
):
    """
    构造 Workflow 无日志监控的 HTML 邮件正文。

    参数:
        monitor_id: 监控实例标识。
        end_time: 本次检查时间。
        no_log_hours: 无日志阈值（小时）。
        step_num_list: 当前监控的 step_num 列表。
    """

    no_log_msg = f"No step log found for the monitored steps in the last {no_log_hours} hours."

    return f"""
    <html>
      <body>
        <h2>Workflow No Log Monitor - {monitor_id}</h2>
        <p>
          Check time: {end_time.isoformat()}<br>
          Project: {DEFAULT_PROJECT}<br>
          Monitored step_nums: {', '.join(step_num_list)}<br>
          No-log threshold: {no_log_hours} hours
        </p>

        <h3>No Log Alert (>{no_log_hours}h)</h3>
        <p>{no_log_msg}</p>
      </body>
    </html>
    """


In [0]:
def monitor_main(
    monitor_id: str,
    log_table: str,
    project: str,
    step_num_list: list[str],
    no_log_hours: int,
    end_time: datetime,
    to_addrs=None,
):
    """
    Workflow 无日志监控主入口。

    逻辑:
        1. 查询窗口取 no_log_hours，与告警阈值保持一致。
        2. 读取 t_task_step_log 中 project 和 step_num 命中且 end_time 在窗口内的记录。
        3. 调用无日志检测，命中则发送邮件。

    参数:
        monitor_id: 监控实例标识，用于邮件主题和日志。
        log_table: t_task_step_log 表名（可带 catalog/database 前缀）。
        project: 要监控的 project 字段值。
        step_num_list: 监控的 step_num 列表。
        no_log_hours: 无日志告警阈值小时数（默认 2），同时作为查询窗口时长。
        end_time: 本次检查基准时间。
        to_addrs: 收件人列表，为空时使用 TO_ADDRS。
    """
    start_time = end_time - timedelta(hours=no_log_hours)

    print(f"monitor_id: {monitor_id}")
    print(f"log_table: {log_table}")
    print(f"project: {project}")
    print(f"step_num_list: {step_num_list}")
    print(f"check window: {start_time} -> {end_time}")
    print(f"to_addrs: {to_addrs}")

    if not step_num_list:
        raise ValueError("step_num_list cannot be empty")

    # 读取并过滤 step log：限定 project、指定 step_num、指定时间窗口。
    log_df = (
        spark.table(log_table)
            .filter(F.col("project") == project)
            .filter(F.col("step_num").isin(step_num_list))
            .filter((F.col("end_time") >= F.lit(start_time)) & (F.col("end_time") < F.lit(end_time)))
    )

    has_no_log_alert = log_df.limit(1).count() == 0

    if has_no_log_alert:
        print(f"No log alert: {monitor_id}")

        html_body = build_email_body(
            monitor_id=monitor_id,
            end_time=end_time,
            no_log_hours=no_log_hours,
            step_num_list=step_num_list,
        )

        recipients = to_addrs or TO_ADDRS
        if not recipients:
            raise ValueError("to_addrs is empty; no recipients configured for the workflow no log monitor email.")

        send_email(
            subject=SUBJECT.format(yyyymmdd=end_time.strftime("%Y%m%d")),
            html_body=html_body,
            to_addrs=recipients,
            cc_addrs=CC_ADDRS,
            bcc_addrs=BCC_ADDRS,
            custom_text="",
        )
        print(f"No log alert email sent: {monitor_id}")
    else:
        print(f"No alert for this check: {monitor_id}")


In [0]:
# 邮件收件人配置。
TO_ADDRS: list[str] = []
CC_ADDRS: list[str] = []
BCC_ADDRS: list[str] = []

SUBJECT = "[Critical] [MDM] Job No Log Alert {yyyymmdd}"

In [0]:
# 从 Databricks Widget 读取运行参数；缺失时使用默认值。
monitor_id = dbutils.widgets.get("monitor_id")
log_table = dbutils.widgets.get("log_table")
project = dbutils.widgets.get("project") or DEFAULT_PROJECT
step_num_list_str = dbutils.widgets.get("step_num_list")
step_num_list = parse_step_num_list(step_num_list_str)

no_log_hours = int(dbutils.widgets.get("no_log_hours") or DEFAULT_NO_LOG_HOURS)

# trigger_timestamp_ms 由调度任务传入（毫秒级 Unix 时间戳），
# 用于统一本次检查的基准时间，避免各节点取本地 now() 造成漂移。
try:
    trigger_timestamp_ms = int(dbutils.widgets.get("trigger_timestamp_ms")) / 1000
except Exception:
    trigger_timestamp_ms = int(datetime.now().timestamp())

# Comma-separated list of recipient email addresses.
to_addrs_str = dbutils.widgets.get("to_addrs")
to_addrs = [x.strip() for x in to_addrs_str.split(",") if x.strip()] if to_addrs_str else TO_ADDRS

end_time = datetime.fromtimestamp(trigger_timestamp_ms)

monitor_main(
    monitor_id=monitor_id,
    log_table=log_table,
    project=project,
    step_num_list=step_num_list,
    no_log_hours=no_log_hours,
    end_time=end_time,
    to_addrs=to_addrs,
)